In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

In [4]:
df = pd.read_csv("data/features.csv")

print(df.head())
print(df.columns)
print(df.shape)
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

   article_id  sentence_id                                           sentence  \
0           0            0  LONDON, England (Reuters) -- Harry Potter star...   
1           0            1  Daniel Radcliffe as Harry Potter in "Harry Pot...   
2           0            2  "I don't plan to be one of those people who, a...   
3           0            3   "I don't think I'll be particularly extravagant.   
4           0            4  "The things I like buying are things that cost...   

   label  relative_position  sentence_length  tfidf_score  named_entity_count  \
0      1               0.00               38     0.025048                   5   
1      1               0.04               43     0.024866                   4   
2      0               0.08               36     0.022746                   0   
3      0               0.12                9     0.011982                   0   
4      0               0.16               17     0.015520                   0   

   first_sentence_overlap 

In [6]:
feature_cols = [
    "relative_position",
    "sentence_length",
    "tfidf_score",
    "named_entity_count",
    "first_sentence_overlap"
]

X = df[feature_cols]
y = df["label"].astype(int)
groups = df["article_id"]

In [7]:
print(X.isna().sum())
print(y.isna().sum())

relative_position         0
sentence_length           0
tfidf_score               0
named_entity_count        0
first_sentence_overlap    0
dtype: int64
0


In [8]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_articles = set(df.iloc[train_idx]["article_id"])
test_articles = set(df.iloc[test_idx]["article_id"])

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train label distribution:")
print(y_train.value_counts(normalize=True))

print("Test label distribution:")
print(y_test.value_counts(normalize=True))

print("Article overlap:", len(train_articles.intersection(test_articles)))

Train shape: (25784, 5)
Test shape: (6825, 5)
Train label distribution:
label
0    0.891987
1    0.108013
Name: proportion, dtype: float64
Test label distribution:
label
0    0.896264
1    0.103736
Name: proportion, dtype: float64
Article overlap: 0


In [9]:
dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

print("Dummy Model Results")
print("Accuracy:", accuracy_score(y_test, dummy_preds))
print("Precision:", precision_score(y_test, dummy_preds, zero_division=0))
print("Recall:", recall_score(y_test, dummy_preds, zero_division=0))
print("F1:", f1_score(y_test, dummy_preds, zero_division=0))
print(confusion_matrix(y_test, dummy_preds))
print(classification_report(y_test, dummy_preds, zero_division=0))

Dummy Model Results
Accuracy: 0.8962637362637362
Precision: 0.0
Recall: 0.0
F1: 0.0
[[6117    0]
 [ 708    0]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6117
           1       0.00      0.00      0.00       708

    accuracy                           0.90      6825
   macro avg       0.45      0.50      0.47      6825
weighted avg       0.80      0.90      0.85      6825



In [10]:
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

log_reg.fit(X_train, y_train)

log_preds = log_reg.predict(X_test)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, log_preds))
print("Precision:", precision_score(y_test, log_preds, zero_division=0))
print("Recall:", recall_score(y_test, log_preds, zero_division=0))
print("F1:", f1_score(y_test, log_preds, zero_division=0))
print(confusion_matrix(y_test, log_preds))
print(classification_report(y_test, log_preds, zero_division=0))

Logistic Regression Results
Accuracy: 0.7346520146520147
Precision: 0.22684497275879148
Recall: 0.6468926553672316
F1: 0.3359002566923359
[[4556 1561]
 [ 250  458]]
              precision    recall  f1-score   support

           0       0.95      0.74      0.83      6117
           1       0.23      0.65      0.34       708

    accuracy                           0.73      6825
   macro avg       0.59      0.70      0.59      6825
weighted avg       0.87      0.73      0.78      6825



In [ ]:
models = {
    "Dummy Majority": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB())
    ]),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=10000
        ))
    ])
}

In [12]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)

    results.append({
        "model": name,
        "accuracy": acc,
        "precision_label_1": prec,
        "recall_label_1": rec,
        "f1_label_1": f1
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="f1_label_1", ascending=False))

                  model  accuracy  precision_label_1  recall_label_1  \
2         Random Forest  0.900073           0.538690        0.255650   
4            Linear SVM  0.734799           0.227767        0.651130   
1   Logistic Regression  0.734652           0.226845        0.646893   
3  Gaussian Naive Bayes  0.875311           0.326034        0.189266   
0        Dummy Majority  0.896264           0.000000        0.000000   

   f1_label_1  
2    0.346743  
4    0.337482  
1    0.335900  
3    0.239500  
0    0.000000  


In [15]:
best_model = models["Linear SVM"]

best_model.fit(X_train, y_train)
best_preds = best_model.predict(X_test)

print(classification_report(y_test, best_preds, zero_division=0))
print(confusion_matrix(y_test, best_preds))

              precision    recall  f1-score   support

           0       0.95      0.74      0.83      6117
           1       0.23      0.65      0.34       708

    accuracy                           0.73      6825
   macro avg       0.59      0.70      0.59      6825
weighted avg       0.87      0.73      0.78      6825

[[4554 1563]
 [ 247  461]]


In [18]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

for name, model in models.items():
    cv_results = cross_validate(
        model,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    print("\n", name)
    for metric in scoring.keys():
        scores = cv_results[f"test_{metric}"]
        print(metric, "mean:", round(scores.mean(), 4), "std:", round(scores.std(), 4))

/home/raksha/code/yt_video_summarizer/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/raksha/code/yt_video_summarizer/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/raksha/code/yt_video_summarizer/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri


 Dummy Majority
accuracy mean: 0.8929 std: 0.0006
precision mean: 0.0 std: 0.0
recall mean: 0.0 std: 0.0
f1 mean: 0.0 std: 0.0

 Logistic Regression
accuracy mean: 0.7192 std: 0.0036
precision mean: 0.2286 std: 0.003
recall mean: 0.6828 std: 0.009
f1 mean: 0.3425 std: 0.0042

 Random Forest
accuracy mean: 0.8937 std: 0.0013
precision mean: 0.5075 std: 0.0094
recall mean: 0.2539 std: 0.0073
f1 mean: 0.3385 std: 0.0079

 Gaussian Naive Bayes
accuracy mean: 0.8708 std: 0.003
precision mean: 0.339 std: 0.0124
recall mean: 0.2164 std: 0.0142
f1 mean: 0.2639 std: 0.0109

 Linear SVM
accuracy mean: 0.7206 std: 0.0035
precision mean: 0.2292 std: 0.0042
recall mean: 0.6805 std: 0.0127
f1 mean: 0.3429 std: 0.0062


In [19]:
log_reg.fit(X_train, y_train)

probs = log_reg.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:
    preds = (probs >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "num_predicted_important": preds.sum()
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df)

    threshold  precision    recall        f1  num_predicted_important
0        0.10   0.105515  0.997175  0.190837                     6691
1        0.15   0.111985  0.985876  0.201124                     6233
2        0.20   0.120939  0.967514  0.215003                     5664
3        0.25   0.131317  0.932203  0.230206                     5026
4        0.30   0.143410  0.889831  0.247010                     4393
5        0.35   0.155585  0.826271  0.261862                     3760
6        0.40   0.176095  0.778249  0.287204                     3129
7        0.45   0.198601  0.721751  0.311490                     2573
8        0.50   0.226845  0.646893  0.335900                     2019
9        0.55   0.254353  0.577684  0.353195                     1608
10       0.60   0.288889  0.514124  0.369919                     1260
11       0.65   0.329854  0.446328  0.379352                      958
12       0.70   0.357759  0.351695  0.354701                      696
13       0.75   0.38

In [20]:
chosen_threshold = 0.45

final_preds = (probs >= chosen_threshold).astype(int)

print(classification_report(y_test, final_preds, zero_division=0))
print(confusion_matrix(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.95      0.66      0.78      6117
           1       0.20      0.72      0.31       708

    accuracy                           0.67      6825
   macro avg       0.58      0.69      0.55      6825
weighted avg       0.88      0.67      0.73      6825

[[4055 2062]
 [ 197  511]]


In [21]:
best_model = log_reg
best_model.fit(X, y)

joblib.dump(best_model, "sentence_importance_model.joblib")

print("Saved model as sentence_importance_model.joblib")

Saved model as sentence_importance_model.joblib


In [22]:
joblib.dump(chosen_threshold, "sentence_importance_threshold.joblib")

['sentence_importance_threshold.joblib']